In [9]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [2]:
load_dotenv()
#DATA_DIR = os.environ["DATA_DIR"]
# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")
# Define Forecast Months
FC_MONTHS = [9, 10, 11, 12, 1, 2]
FC_PERIOD = slice("2026-09-01", "2027-02-28")

config = analysis_utils.ImpactConfig( version = "v260910",
                                     baseline_period=BASELINE_PERIOD, 
                                     polygons_path= os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
                                     socioeconomics_path= os.environ["POREALLAS_SOCIOECONOMICS_URI"],
                                     rate=False, 
                                     months = FC_MONTHS,
                                     hotonly = "hotonly", 
                                     dims = ['number', 'sample'])

# Define Forecast
EFFECTS_URI = "/home/emily_zuetell/projects/poreallas/data/v20260909_effects_with_betas.zarr" # Can be any effects datatree

In [4]:
# Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [5]:
### Log baseline period and impact calculation
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

In [10]:
# Compute impact: forecast - baseline
impact = config.compute_impact(effect.chunk({dim: -1 for dim in config.dims}), ensemble=True) #Maintain individual ensemble members
# Use only the defined 6-months
impact = impact.sel(month=config.months)

In [11]:
# Aggregate Impact Regions to group_level
group_level = 'ADM1' #IR: Impact region, # ADM1: State level, # ISO: Country level
# Use a subset of ensemble members for quick testing (.sel(number = ...))
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact.sel(number = [0, 1, 2, 3, 4]), config, group_level)

source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped
source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped


In [12]:
#Compute stats in xarray from dims in config.dims
stat_cols = ["median", "p17", "p83", "likely_range_IPCC", "mean", "std", "min", "max", "p10", "p90"]
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))

In [13]:
# Format dataframe for csv output       
wide = _polygons_impact.pivot(
    index=base_cols,
    columns="month", values=stat_cols,
)
wide.columns = [f"month {m} {stat}" for stat, m in wide.columns]
stat_col_names = wide.columns.difference(base_cols)
wide[stat_col_names] = wide[stat_col_names].round(0).astype("Int64")
wide = wide.reset_index()

In [14]:
wide

,GID_0,GID_1,month 1 median,month 2 median,month 9 median,month 10 median,month 11 median,month 12 median,month 1 p17,month 2 p17,...,month 9 p10,month 10 p10,month 11 p10,month 12 p10,month 1 p90,month 2 p90,month 9 p90,month 10 p90,month 11 p90,month 12 p90
0,ABW,ABW,0,0,0,0,0,0,0,0,...,-1,0,0,0,1,1,0,0,1,1
1,AFG,AFG.10_1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,AFG,AFG.11_1,0,0,14,-1,0,0,0,0,...,-9,-7,-2,0,0,0,38,29,0,0
3,AFG,AFG.12_1,0,0,-4,0,0,0,0,0,...,-19,-3,-2,0,0,0,4,7,0,0
4,AFG,AFG.13_1,0,0,-3,0,0,0,0,0,...,-9,0,-1,0,0,0,4,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3678,ZWE,ZWE.5_1,18,11,-3,-3,9,4,5,3,...,-11,-20,-16,-9,45,39,-1,22,40,48
3679,ZWE,ZWE.6_1,21,16,-6,-8,20,11,7,6,...,-18,-30,-17,-12,62,46,-2,49,77,68
3680,ZWE,ZWE.7_1,38,18,-7,-5,0,20,12,6,...,-20,-22,-12,-11,131,66,-2,4,33,66
3681,ZWE,ZWE.8_1,29,16,-5,-8,16,15,10,7,...,-15,-26,-10,-7,79,42,-2,22,68,70


In [ ]:
# Output CSV

# Log parameters in filename
filename_template="{version}_{hotonly}_{scope}_{rate_l}_{stat_scope}_{group_level}_{baseline}.csv"

wide.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
    ),
    index=False,
)